# 1 — Buck-Boost Converter Modeling

> **Goal.** Derive the small-signal model of the inverting buck-boost
> in CCM, show how the **polarity inversion** falls out of the
> topology rather than the math, and identify the **RHP zero with
> duty-cycle-dependent location** — the buck-boost's biggest control
> design challenge.

**Prerequisites**

- Buck and boost reference notebooks (`projects/converters/{buck,boost}/`).
  The state-space averaging procedure is the same; only the topology
  changes.

**What you'll be able to do at the end**

1. Write the switched model for both ON and OFF intervals of the
   inverting buck-boost.
2. Derive the steady-state ratio $|V_o| = D \\cdot V_g / (1 - D)$.
3. Identify where in the algebra the polarity inversion appears.
4. Locate the RHP zero in $G_{vd}(s)$ and explain why it depends on
   $1/D$ (so high step-up ratios slow the loop more).
5. Compare the buck-boost's three characteristic regimes (buck-like
   $D < 0.5$, balanced $D = 0.5$, boost-like $D > 0.5$).


## Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
from scipy import signal
import matplotlib.pyplot as plt

from buck_boost_model import (
    BuckBoostParams,
    buck_boost_state_space,
    control_to_output_tf,
    line_to_output_tf,
    output_impedance_tf,
    operating_point_report,
)

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


## 1. The inverting buck-boost topology

```
   V_g ----- S -----+
                    |
                    +---L---+
                    |       |
                    +       gnd
                    |
            D (anode toward L node, cathode toward V_o node)
                    |
                    +---+---+--- V_o  (V_o < 0)
                    |   |   |
                    C   R  load
                    |   |   |
                   gnd gnd gnd
```

- The switch $S$ chops $V_g$ into a pulsed waveform at node $A$ (top
  of $L$).
- During ON, $L$ stores energy from the input (the diode is off).
- During OFF, $S$ opens; the inductor maintains its current by
  forward-biasing $D$ and pumping charge into the output cap. The
  geometry forces the cap's $V_o$-side to go NEGATIVE relative to
  ground — that's where the polarity inversion comes from.

### 1.1 Operating-point intuition

In steady state:

- Average inductor voltage = 0 → $D \\cdot V_g = (1-D) \\cdot |V_o|$
  → $\\boxed{|V_o| = \\dfrac{D}{1 - D} V_g}$ (with negative polarity).
- $D = 0.5$: $|V_o| = V_g$ exactly.
- $D < 0.5$: $|V_o| < V_g$ (buck-like step-down).
- $D > 0.5$: $|V_o| > V_g$ (boost-like step-up). As $D \\to 1$,
  $|V_o| \\to \\infty$ (parasitic losses limit in practice).


## 2. Switched (instantaneous) model

States: $i_L$ (inductor current, positive when flowing into the
top of $L$) and $v_o$ (output cap voltage MAGNITUDE — we'll write
all equations using positive $v_o$ and remember the actual polarity
is negative).

### 2.1 ON interval ($S$ closed, $D$ off)

Inductor sees the full input bus. Output cap drains through the load:

$$
L \\frac{di_L}{dt} = v_g,
\\qquad
C \\frac{dv_o}{dt} = -\\frac{v_o}{R}
$$

### 2.2 OFF interval ($S$ open, $D$ on)

Inductor's current is now forced through the diode into the cap. The
inductor's voltage flips:

$$
L \\frac{di_L}{dt} = -v_o,
\\qquad
C \\frac{dv_o}{dt} = i_L - \\frac{v_o}{R}
$$

Compare to the boost: the OFF inductor equation has $-v_o$ instead of
$(v_g - v_o)$. That's the topological difference — the buck-boost's
inductor is grounded at the source end during OFF, not connected to
$v_g$.


## 3. State-space averaging

Let $q(t) \\in \\{0, 1\\}$ be the switching function. Continuous form:

$$
L \\frac{di_L}{dt} = q \\cdot v_g + (1 - q) \\cdot (-v_o) = q v_g - (1-q) v_o
$$

$$
C \\frac{dv_o}{dt} = q \\cdot \\left(-\\frac{v_o}{R}\\right)
                   + (1-q)\\left(i_L - \\frac{v_o}{R}\\right)
                   = (1-q) i_L - \\frac{v_o}{R}
$$

Replace $q \\to d$ for the average (assumes $f_{sw} \\gg f_n$):

$$\\boxed{
\\;\\; L \\frac{di_L}{dt} = d v_g - (1 - d) v_o
\\;\\;}
$$

$$\\boxed{
\\;\\; C \\frac{dv_o}{dt} = (1 - d) i_L - \\frac{v_o}{R}
\\;\\;}
$$

The cap equation is IDENTICAL to the boost's — same `(1-d)·i_L` source
term and same `-v_o/R` load. The inductor equation differs only in the
ON-state input ($d \\cdot v_g$ in buck-boost; $v_g$ in boost).


### 3.1 Steady-state

$$
0 = D V_g - (1-D) V_o \\;\\implies\\; |V_o| = \\frac{D}{1-D} V_g
$$

$$
0 = (1-D) I_L - \\frac{V_o}{R} \\;\\implies\\; I_L = \\frac{V_o}{R(1-D)}
$$

The inductor-current formula matches the boost's exactly.


In [ ]:
params = BuckBoostParams()
print(operating_point_report(params))


## 4. Small-signal linearization

Perturb $i_L = I_L + \\hat i_L$, $v_o = V_o + \\hat v_o$, $d = D + \\hat d$,
$v_g = V_g + \\hat v_g$, substitute, drop products:

From the inductor equation:

$$
L \\frac{d\\hat i_L}{dt}
   = D \\hat v_g + (V_g + V_o) \\hat d - (1-D) \\hat v_o
$$

Note $V_g + V_o = V_g / (1-D)$ at the operating point — a useful
substitution that highlights the DC-gain dependence.

From the cap equation:

$$
C \\frac{d\\hat v_o}{dt}
   = (1-D) \\hat i_L - I_L \\hat d - \\frac{\\hat v_o}{R}
$$

Same shape as the boost: the $-I_L \\hat d$ term in the cap equation
is what gives both topologies their RHP zero.


## 5. State-space matrices

$$
A = \\begin{bmatrix}
0 & -(1-D)/L \\\\
(1-D)/C & -1/(R C)
\\end{bmatrix},
\\quad
B = \\begin{bmatrix}
(V_g + V_o)/L & D/L \\\\
-I_L/C & 0
\\end{bmatrix}
$$

$A$ is IDENTICAL to the boost's. $B$ differs only in element $B_{00}$:
- Boost: $B_{00} = V_o / L$
- Buck-boost: $B_{00} = (V_g + V_o) / L$ — bigger gain because the
  inductor effectively sees the combined input+output voltage when
  duty perturbs.


In [ ]:
A, B, C_mat, D_mat = buck_boost_state_space(params)
print("A ="); print(A); print()
print("B = [col 0: d̂   col 1: v̂_g]"); print(B); print()
print("C =", C_mat)
print("D (feedthrough) =", D_mat)
print()
eig = np.linalg.eigvals(A)
print(f"A eigenvalues: {eig}")
print(f"Pole magnitude (= ω_n): {abs(eig[0]):.1f} rad/s  "
      f"(expect {params.omega_n:.1f})")


## 6. Transfer functions

### 6.1 $G_{vd}(s)$ — the headline plant

Closed form (Erickson Eq 8.55, taking magnitudes):

$$
G_{vd}(s) = \\frac{V_g}{(1-D)^2} \\cdot
\\frac{1 - s/\\omega_{z,RHP}}{1 + s/(Q \\omega_n) + (s/\\omega_n)^2}
$$

with:

| Quantity | Buck-boost | Boost |
|---|---|---|
| DC gain magnitude | $V_g / (1-D)^2$ | $V_o / (1-D) = V_g/(1-D)^2$ |
| $\\omega_n$ | $(1-D)/\\sqrt{LC}$ | $(1-D)/\\sqrt{LC}$ |
| $Q$ | $(1-D) R \\sqrt{C/L}$ | $(1-D) R \\sqrt{C/L}$ |
| $\\omega_{z,RHP}$ | $\\dfrac{R(1-D)^2}{L \\cdot D}$ | $\\dfrac{R(1-D)^2}{L}$ |

**Key difference**: the buck-boost RHP zero has an extra $1/D$ factor.
At $D = 0.5$ they're equal; at $D = 0.8$ the buck-boost zero is at
**1/4** of the boost's (much harder to control). At $D = 0.2$
(step-down regime) it's 5× **higher** (easier).

### 6.2 $G_{vg}(s)$ and $Z_{out}(s)$

$$
G_{vg}(s) = \\frac{D/(1-D)}{1 + s/(Q \\omega_n) + (s/\\omega_n)^2},
\\quad
Z_{out}(s) = \\frac{sL/(1-D)^2}{1 + s/(Q \\omega_n) + (s/\\omega_n)^2}
$$

DC line-to-output gain is $D/(1-D)$ = the duty-ratio gain magnitude
itself. No RHP zeros in either of these (only $G_{vd}$ has it).


In [ ]:
Gvd = control_to_output_tf(params)
Gvg = line_to_output_tf(params)
Zout = output_impedance_tf(params)

print(f"Gvd(0)  (mag)   = {Gvd.num[1] / Gvd.den[2]:.3f} V/duty  "
      f"(expect V_g/(1-D)² = {params.V_g/(1-params.D)**2:.3f})")
print(f"Gvg(0)          = {Gvg.num[0] / Gvg.den[2]:.4f} V/V       "
      f"(expect D/(1-D)   = {params.D/(1-params.D):.4f})")
print()
print(f"Gvd zeros: {np.roots(Gvd.num)}  "
      f"(expect RHP at +{params.omega_z_rhp:.0f} rad/s = "
      f"+{params.f_z_rhp:.0f} Hz)")
print(f"Gvd poles: {np.roots(Gvd.den)}  (expect Q-LC pair)")


### 6.3 Bode plots — compare buck-boost vs boost RHP zeros

Plot $|G_{vd}|$ and phase. The natural pole pair is at the same
location as the boost (since $\\omega_n$ depends only on $L$, $C$,
$(1-D)$), but the RHP zero is at higher frequency for $D = 0.5$.


In [ ]:
f = np.logspace(0, np.log10(params.f_sw), 1500)
w = 2 * np.pi * f

fig, (ax_mag, ax_ph) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)
for tf, name, style in [
    (Gvd,  r"$G_{vd}(s)$  control → output (mag)", "-"),
    (Gvg,  r"$G_{vg}(s)$  line → output",          "--"),
    (Zout, r"$Z_{out}(s)$ load → output",          ":"),
]:
    _, mag, ph = signal.bode(tf, w=w)
    ax_mag.semilogx(f, mag, style, label=name)
    ax_ph.semilogx(f, ph, style, label=name)

for ax in (ax_mag, ax_ph):
    ax.axvline(params.f_n,     color="C0", linestyle=":", alpha=0.4,
               label=f"$f_n$ = {params.f_n:.0f} Hz")
    ax.axvline(params.f_z_rhp, color="C3", linestyle=":", alpha=0.6,
               label=f"$f_{{z,RHP}}$ = {params.f_z_rhp:.0f} Hz")
    ax.legend(loc="best", fontsize=8)

ax_mag.set_ylabel("Magnitude [dB]")
ax_ph.set_ylabel("Phase [deg]")
ax_ph.set_xlabel("Frequency [Hz]")
ax_mag.set_title(f"Buck-boost open-loop "
                 f"($V_g$={params.V_g}V, $|V_o|$={params.V_o}V, "
                 f"D={params.D:.2f})")
plt.tight_layout()
plt.show()


## 7. The wrong-way step response

Apply a small duty step. Like the boost, the buck-boost's $G_{vd}$
has an RHP zero, so the output **dips before rising** on a positive
duty step.


In [ ]:
duty_step = 0.01
t = np.linspace(0, 10e-3, 5000)
_, y_step = signal.step(Gvd, T=t)
v_o_pred = params.V_o + duty_step * y_step  # magnitude

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(t*1e3, v_o_pred, label="$|v_o|$ (analytical)")
ax.axhline(params.V_o, color="k", linestyle=":", alpha=0.4,
           label=f"Pre-step $|V_o|$ = {params.V_o} V")
expected_new = params.V_o + duty_step * params.V_g / (1 - params.D)**2
ax.axhline(expected_new, color="g", linestyle=":", alpha=0.5,
           label=f"Predicted new $|V_o|$ ≈ {expected_new:.3f} V")
ax.set_xlabel("Time [ms]")
ax.set_ylabel("$|v_o|$ [V]")
ax.set_title(f"Buck-boost response to a {duty_step*100:.0f}% duty step "
             "(notice the RHP-zero dip first)")
ax.legend()
plt.tight_layout()
plt.show()

dip = params.V_o - np.min(v_o_pred)
print(f"Pre-step $|V_o|$ = {params.V_o:.4f} V")
print(f"Initial dip      = {dip*1e3:.1f} mV "
      f"({dip/params.V_o*100:.2f} % of $|V_o|$)")
print(f"Final $|v_o|$    = {v_o_pred[-1]:.4f} V")


## 8. Model self-consistency checks

Same three rigorous checks as the buck and boost notebooks — pure math,
no simulator involved.


In [ ]:
A, B, C_mat, D_mat = buck_boost_state_space(params)
Gvd_closed = control_to_output_tf(params)

# (1) Poles
ss_poles = sorted(np.linalg.eigvals(A), key=lambda z: z.imag)
tf_poles = sorted(np.roots(Gvd_closed.den), key=lambda z: z.imag)
print("(1) Poles:")
print(f"    SS:  {ss_poles}")
print(f"    TF:  {tf_poles}")
pole_match = np.allclose(ss_poles, tf_poles, rtol=1e-10)
print(f"    → match: {pole_match}")

# (2) DC gains
print()
print("(2) DC gains:")
print(f"    Gvd(0)  = {Gvd_closed.num[1] / Gvd_closed.den[2]:8.4f}  "
      f"(expect V_g/(1-D)² = {params.V_g/(1-params.D)**2:.4f})")
Gvg = line_to_output_tf(params)
print(f"    Gvg(0)  = {Gvg.num[0] / Gvg.den[2]:8.4f}  "
      f"(expect D/(1-D) = {params.D/(1-params.D):.4f})")

# (3) ss2tf round-trip
num_from_ss, den_from_ss = signal.ss2tf(A, B, C_mat, D_mat, input=0)
num_from_ss = np.trim_zeros(num_from_ss.flatten(), trim='f')
scale_ss = den_from_ss[0]
scale_cf = Gvd_closed.den[0]
num_ss_norm = num_from_ss / scale_ss
num_cf_norm = np.array(Gvd_closed.num) / scale_cf
den_ss_norm = np.array(den_from_ss) / scale_ss
den_cf_norm = np.array(Gvd_closed.den) / scale_cf
print()
print("(3) ss2tf round-trip (normalized):")
print(f"    Closed-form num = {num_cf_norm}")
print(f"    From-SS num     = {num_ss_norm}")
print(f"    Closed-form den = {den_cf_norm}")
print(f"    From-SS den     = {den_ss_norm}")
round_trip_ok = (
    np.allclose(num_ss_norm, num_cf_norm, rtol=1e-9)
    and np.allclose(den_ss_norm, den_cf_norm, rtol=1e-9)
)
print(f"    → match: {round_trip_ok}")

assert pole_match and round_trip_ok, "Self-consistency check failed!"
print()
print("✅  All three self-consistency checks pass.")


## 9. The "RHP zero migrates with duty" experiment

Re-derive the operating point for several duties and plot the RHP zero
frequency $f_{z,RHP}$ as a function of $D$. Watch it move dramatically:
at $D = 0.2$ the zero is at ~25 kHz; at $D = 0.8$ it crashes to under
2 kHz. That's the bandwidth ceiling for any voltage-mode controller.


In [ ]:
D_grid = np.linspace(0.05, 0.95, 100)
f_z_grid = []
f_n_grid = []
V_o_grid = []
for d in D_grid:
    V_o_d = d / (1 - d) * params.V_g
    p_d = BuckBoostParams(V_g=params.V_g, V_o=V_o_d, R=params.R,
                          L=params.L, C=params.C, f_sw=params.f_sw)
    f_z_grid.append(p_d.f_z_rhp)
    f_n_grid.append(p_d.f_n)
    V_o_grid.append(V_o_d)

fig, (ax_v, ax_f) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax_v.plot(D_grid, V_o_grid, "C0", linewidth=2,
          label="|V_o| (output magnitude)")
ax_v.axhline(params.V_g, color="k", linestyle=":", alpha=0.4,
             label=f"V_g = {params.V_g} V")
ax_v.set_ylabel("|V_o| [V]")
ax_v.set_ylim(0, 60)
ax_v.legend()
ax_v.set_title("Buck-boost: |V_o| and the RHP zero vs duty cycle")

ax_f.semilogy(D_grid, f_z_grid, "C3", linewidth=2,
              label=r"$f_{z,RHP}$ (RHP zero)")
ax_f.semilogy(D_grid, f_n_grid, "C0", linestyle="--",
              label=r"$f_n$ (LC double pole)")
ax_f.axhline(params.f_sw/10, color="k", linestyle=":", alpha=0.4,
             label=f"$f_{{sw}}/10$ = {params.f_sw/10/1e3:.0f} kHz "
                   "(buck-style bandwidth cap)")
ax_f.axvline(0.5, color="g", linestyle=":", alpha=0.4,
             label="D = 0.5 (|V_o| = V_g)")
ax_f.set_xlabel("Duty cycle D")
ax_f.set_ylabel("Frequency [Hz]")
ax_f.legend(loc="best")
plt.tight_layout()
plt.show()

print("Notice three regimes:")
print(f"  D=0.2 (buck-like): f_z = {f_z_grid[np.argmin(np.abs(D_grid-0.2))]/1e3:.1f} kHz")
print(f"  D=0.5 (balanced):  f_z = {f_z_grid[np.argmin(np.abs(D_grid-0.5))]/1e3:.1f} kHz")
print(f"  D=0.8 (step-up):   f_z = {f_z_grid[np.argmin(np.abs(D_grid-0.8))]/1e3:.1f} kHz")


## 10. Summary

The inverting buck-boost combines the buck's and boost's modeling
machinery. State-space averaging produces a clean small-signal model
where:

- The polarity inversion is a TOPOLOGY detail, not a math artifact;
  we model $|V_o|$ as positive and add a sign at the physical layout.
- The same $-I_L \\hat d$ term in the capacitor equation that gave
  the boost its RHP zero also appears here — so the buck-boost is
  also non-minimum-phase.
- The RHP zero location $\\omega_{z,RHP} = R(1-D)^2 / (L D)$ depends
  *inversely* on $D$. At high duty (high step-up) the zero crashes to
  low frequency, capping the closed-loop bandwidth tighter.

Math validated by three checks (poles, DC gains, ss2tf round-trip).

**Next**: open `02_buck_boost_controller.ipynb` to size a Type-III
compensator with $f_c \\le f_{z,RHP}/5$, discretize via Tustin, and
run a switched closed-loop simulation that proves the design tracks
a reference step at the chosen duty.

**Note on Pulsim cross-validation.** Pulsim's switching engine has
the same numerical-stability issues with the ideal-switch buck-boost
that we saw with the boost (decoupled output cap → unbounded
switching-transition dV/dt). The math self-consistency above is the
rigorous validation; the closed-loop pure-Python simulation in
notebook 2 demonstrates the controller working on the real switched
waveform.

**Suggested exercises**

1. Set $V_o = 48$ V (D = 0.8) and re-derive the operating point and
   $f_{z,RHP}$. How much does the bandwidth cap tighten?
2. Set $V_o = 6$ V (D = 0.33) for a true step-DOWN buck-boost.
   How does the RHP zero compare to the boost's at the same R, L, C?
3. Derive the SEPIC topology (4th-order, non-inverting buck-boost).
   Where do its TWO RHP zeros come from?
